# Preparación del dataset BOSSBase con Matrix Embedding usando códigos de Hamming

Este notebook construye el corpus experimental usado para entrenar la red neuronal de esteganálisis. 
La preparación incluye la descarga de BOSSBase, la generación de imágenes esteganográficas mediante Matrix Embedding con códigos de Hamming (7,4) y (15,11), la creación de particiones sin data leakage y la evaluación de fidelidad visual mediante PSNR y SSIM.

Este notebook se ejecuta únicamente cuando se desea construir o reconstruir el dataset. 
El entrenamiento de la red neuronal se realiza en el notebook `hamming_embedding_resnet_steganalysis.ipynb`.

## 1. Importaciones

Este notebook está pensado para ejecutarse en un `.ipynb` normal. La descarga se realiza con `urllib.request`.

In [2]:
import os
import re
import cv2
import json
import time
import glob
import shutil
import zipfile
import hashlib
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from skimage.metrics import peak_signal_noise_ratio as compute_psnr
from skimage.metrics import structural_similarity as compute_ssim

## 2. Configuración del dataset

Los valores de `VALIDATION_SPLIT`, `TEST_SPLIT` y `SEMILLA` son los siguientes:

```python
VALIDATION_SPLIT = 0.25
TEST_SPLIT = 0.15
SEMILLA = 42
```

estos valores deben ser coherentes con los presentes en el notebook `hamming_embedding_resnet_steganalysis.ipynb`. 

Con `CANTIDAD_A_PROCESAR = 1000`, el split esperado es aproximadamente:

- Train: 600 imágenes base.
- Val: 250 imágenes base.
- Test: 150 imágenes base.

Cada imagen base genera tres archivos asociados dentro de `dataset/`: una cover original, una stego con Hamming (7,4) y una stego con Hamming (15,11). La cantidad de imagenes que se usan en cada conjunto es relevante para entrenar la red neuronal propuesta, que se encuentra en `hamming_embedding_resnet_steganalysis.ipynb`. 

In [ ]:
# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================

URL_BOSSBASE = "http://dde.binghamton.edu/download/ImageDB/BOSSbase_1.01.zip"

# Carpeta raíz del proyecto.
RUTA_PROYECTO = Path(".")

# Archivos y carpetas de entrada
ZIP_PATH = RUTA_PROYECTO / "bossbase.zip"
EXTRACT_DIR = RUTA_PROYECTO / "bossbase_completo"

# Carpetas de salida compatibles con el notebook de entrenamiento
DATASET_DIR = RUTA_PROYECTO / "dataset"
COVER_DIR = DATASET_DIR / "cover"
STEGO_7_DIR = DATASET_DIR / "stego_7"
STEGO_15_DIR = DATASET_DIR / "stego_15"

# split
SPLIT_DIR = RUTA_PROYECTO / "splits"
TRAIN_SPLIT_PATH = SPLIT_DIR / "train_base_names.txt"
VAL_SPLIT_PATH = SPLIT_DIR / "val_base_names.txt"
TEST_SPLIT_PATH = SPLIT_DIR / "test_base_names.txt"

# Salidas de análisis
METADATA_PATH = RUTA_PROYECTO / "dataset_metadata_pytorch.json"
FIDELITY_RESULTS_PATH = RUTA_PROYECTO / "fidelity_results_hamming.csv"

# Parámetros experimentales
CANTIDAD_A_PROCESAR = 1000
VALIDATION_SPLIT = 0.25
TEST_SPLIT = 0.15
SEMILLA = 42

# IMPORTANTE:
# False evita borrar y reconstruir el dataset accidentalmente.
# Cambiar a True solo si se quiere regenerar todo desde cero.
RECONSTRUIR_DATASET = False

# Formato de las imágenes cover.
# "pgm": conserva el formato original de BOSSBase.
# "png": convierte las cover a PNG sin pérdidas.
FORMATO_COVER = "png"

# Cantidad de imágenes usadas para evaluar fidelidad visual.
MUESTRAS_EVALUACION = 100

if FORMATO_COVER not in {"pgm", "png"}:
    raise ValueError("FORMATO_COVER debe ser 'pgm' o 'png'.")

print("Configuración lista.")
print(f"Ruta del proyecto: {RUTA_PROYECTO.resolve()}")
print(f"Dataset de salida: {DATASET_DIR.resolve()}")
print(f"Reconstruir dataset: {RECONSTRUIR_DATASET}")

## 3. Fundamento matemático de Hamming Matrix Embedding

En esta sección se presenta la base matemática usada para generar las imágenes esteganográficas del dataset. 
La teoría de Hamming forma parte del proceso de construcción de los datos, que será relevante para entrenar la red neuronal encontrada en `hamming_embedding_resnet_steganalysis.ipynb`

### 3.1. Código Hamming (7,4)

#### 3.1.1. Implementación de Matrix Embedding con Código Hamming (7,4)

El objetivo del Matrix Embedding es minimizar la cantidad de modificaciones necesarias en la imagen de cobertura, reduciendo así la distorsión estadística. Utilizando un código Hamming (7,4), podemos ocultar 3 bits de información secreta en un bloque de 7 píxeles alterando, como máximo, un solo bit menos significativo (LSB).

Matemáticamente, definimos la matriz de comprobación de paridad $H$ de dimensiones $3 \times 7$. Para un bloque de píxeles $x$, calculamos su síndrome actual. Si la diferencia entre el mensaje secreto y el síndrome actual no es nula, el vector resultante nos indica exactamente el índice de la columna en $H$ (y por ende, el píxel) que debe ser modificado para que el nuevo síndrome coincida con el secreto.

In [ ]:
def embed_hamming_7_3(pixels_block, secret_bits):
    """
    Oculta 3 bits secretos en un bloque de 7 píxeles.
    pixels_block: array de numpy con 7 valores (ej. [150, 151, 149...])
    secret_bits: array de numpy con 3 bits (ej. [1, 0, 1])
    """
    # 1. Matriz de paridad H para Hamming (7,4)
    H = np.array([
        [0, 0, 0, 1, 1, 1, 1],
        [0, 1, 1, 0, 0, 1, 1],
        [1, 0, 1, 0, 1, 0, 1]
    ])

    # 2. Extraer los LSB (Bits Menos Significativos) del bloque de píxeles
    x = pixels_block % 2

    # 3. Calcular el síndrome actual: s_current = H * x^T (mod 2)
    s_current = np.dot(H, x) % 2

    # 4. Calcular la diferencia con el secreto que queremos: d = secret - s_current (mod 2)
    d = (secret_bits - s_current) % 2

    # 5. Encontrar qué píxel debemos cambiar
    # Convertimos el vector 'd' a un número entero (que coincide con la columna en H)
    index_to_change = d[0]*4 + d[1]*2 + d[2]*1

    modified_pixels = pixels_block.copy()

    # Si el índice es 0, la imagen ya tiene el mensaje por pura casualidad (¡No hacemos nada!)
    # Si es > 0, alteramos el LSB del píxel en la posición correspondiente
    if index_to_change > 0:
        idx = index_to_change - 1

        # Invertir el LSB: si es par suma 1, si es impar resta 1
        if modified_pixels[idx] % 2 == 0:
            modified_pixels[idx] += 1
        else:
            modified_pixels[idx] -= 1

    return modified_pixels

#### 3.1.2. Extracción del Mensaje con Código Hamming (7,4)

La principal ventaja de este esquema es la eficiencia en la recuperación. Para extraer el mensaje de la imagen stego, el receptor no necesita conocer la imagen original. Basta con extraer los LSB de los bloques de 7 píxeles y multiplicarlos por la misma matriz de comprobación de paridad $H$. El síndrome resultante será exactamente el mensaje secreto incrustado.

In [ ]:
def extract_hamming_7_3(stego_pixels_block):
    """
    Extrae los 3 bits secretos ocultos en un bloque de 7 píxeles stego.
    stego_pixels_block: array de numpy con 7 valores modificados.
    """
    H = np.array([
        [0, 0, 0, 1, 1, 1, 1],
        [0, 1, 1, 0, 0, 1, 1],
        [1, 0, 1, 0, 1, 0, 1]
    ])

    # 1. Extraer los LSB del bloque stego
    x_stego = stego_pixels_block % 2

    # 2. El mensaje secreto es simplemente el síndrome del bloque
    extracted_bits = np.dot(H, x_stego) % 2

    return extracted_bits

#### 3.1.3. Validación Funcional: Inserción y Extracción con Hamming (7,4)

Para comprobar la correctitud matemática de las operaciones algebraicas de los síndromes, se diseñó una prueba unitaria aislada. Se inyectó el vector secreto [1, 0, 1] en un bloque de control de 7 píxeles.

In [ ]:
# --- Bloque de Prueba Unitaria ---
print("--- Prueba de Funcionamiento ---")
original_pixels = np.array([150, 151, 149, 152, 153, 154, 155])
secret = np.array([1, 0, 1])

print(f"Píxeles originales: {original_pixels}")
print(f"Secreto a ocultar:  {secret}")

# Proceso de embedding
stego_pixels = embed_hamming_7_3(original_pixels, secret)
print(f"Píxeles stego:      {stego_pixels}")

# Proceso de extracción
recovered_secret = extract_hamming_7_3(stego_pixels)
print(f"Secreto recuperado: {recovered_secret}")
print(f"¿Éxito?: {'Sí' if np.array_equal(secret, recovered_secret) else 'No'}")

Como se evidencia en los resultados de ejecución, el algoritmo modificó de manera exitosa un único píxel para satisfacer la matriz de paridad. En términos generales, el procedimiento calculó el índice de modificación a partir del síndrome de diferencia y alteró el LSB del píxel ubicado en esa posición. La prueba también verificó que esta inversión del LSB se realizó sin que el valor del píxel saliera del rango válido [0, 255]. Posteriormente, la función de extracción ciega recuperó el secreto original en su totalidad, validando la integridad del diseño algorítmico y demostrando la eficiencia del Matrix Embedding de primer orden.


### 3.2. Código (15,11)

#### 3.2.1. Escalamiento a Matrix Embedding con Hamming (15,11)

Para analizar un escenario de mayor eficiencia de inserción, implementamos el código Hamming (15,11). En esta configuración, la matriz de comprobación de paridad $H$ tiene dimensiones de $4 \times 15$. Esto nos permite ocultar 4 bits de información en un bloque de 15 píxeles, realizando como máximo una sola modificación.

Teóricamente, esto ofrece un mejor compromiso (trade-off) entre la carga útil y la distorsión estadística, ya que la probabilidad de alterar un píxel disminuye proporcionalmente en bloques más grandes, lo cual evaluaremos frente a las redes residuales.

In [ ]:
def embed_hamming_15_11(pixels_block, secret_bits):
    """
    Oculta 4 bits secretos en un bloque de 15 píxeles usando Matrix Embedding.
    pixels_block: array de numpy con 15 valores (ej. [150, 151, ... 164])
    secret_bits: array de numpy con 4 bits (ej. [1, 0, 1, 1])
    """
    # 1. Matriz H de 4x15. Las columnas representan los números del 1 al 15 en binario.
    H = np.array([
        [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1], # Fila de los 8s
        [0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1], # Fila de los 4s
        [0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1], # Fila de los 2s
        [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1]  # Fila de los 1s
    ])

    x = pixels_block % 2
    s_current = np.dot(H, x) % 2
    d = (secret_bits - s_current) % 2

    # 2. Encontrar el índice a modificar (conversión de 4 bits a decimal)
    index_to_change = d[0]*8 + d[1]*4 + d[2]*2 + d[3]*1

    modified_pixels = pixels_block.copy()

    # 3. Modificar el píxel correspondiente si es necesario
    if index_to_change > 0:
        idx = index_to_change - 1

        if modified_pixels[idx] % 2 == 0:
            modified_pixels[idx] += 1
        else:
            modified_pixels[idx] -= 1

    return modified_pixels

#### 3.2.2. Extracción del Mensaje con Hamming (15,11)

De manera análoga al esquema de menor orden, la recuperación de los datos incrustados mediante Hamming (15,11) se realiza de forma ciega, lo que significa que el receptor no requiere acceso a la imagen de cobertura original para extraer el payload.

El proceso de extracción consiste en segmentar la imagen stego en bloques independientes de 15 píxeles, aislar sus bits menos significativos (LSB) y calcular el producto punto de este vector contra la matriz de comprobación de paridad $H$ de dimensiones $4 \times 15$. El vector resultante de 4 bits corresponde al síndrome del bloque, el cual, por la propiedad del Matrix Embedding, es matemáticamente equivalente al fragmento del mensaje secreto original.

In [ ]:
def extract_hamming_15_11(stego_pixels_block):
    """
    Extrae los 4 bits secretos ocultos en un bloque de 15 píxeles stego.
    """
    H = np.array([
        [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1],
        [0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1],
        [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1]
    ])

    x_stego = stego_pixels_block % 2
    extracted_bits = np.dot(H, x_stego) % 2

    return extracted_bits

#### Validación Funcional: Inserción y Extracción con Hamming (15,11)

De forma equivalente, se evaluó el comportamiento del esquema de mayor orden. Al incrustar un payload de 4 bits ([1, 1, 0, 1]) en un bloque continuo de 15 píxeles, el motor de inserción determinó matemáticamente que solo era necesario alterar un píxel (modificando el valor 132 por 133) para que el nuevo síndrome del bloque coincidiera exactamente con el secreto.


In [ ]:
# --- Bloque de Prueba Unitaria ---
print("--- Prueba Hamming (15,11) ---")
original_pixels_15 = np.array([120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134])
secret_4bits = np.array([1, 1, 0, 1])

stego_pixels_15 = embed_hamming_15_11(original_pixels_15, secret_4bits)
recovered_secret_15 = extract_hamming_15_11(stego_pixels_15)

print(f"Píxeles originales: {original_pixels_15}")
print(f"Secreto a ocultar:  {secret_4bits}")
print(f"Píxeles stego:      {stego_pixels_15}")
print(f"Secreto recuperado: {recovered_secret_15}")
print(f"¿Éxito?: {'Sí' if np.array_equal(secret_4bits, recovered_secret_15) else 'No'}")

La recuperación demostró ser 100% precisa, confirmando que la lógica de prevención de desbordamientos de rango para los LSB funciona sin corromper la información subyacente.

### 3.3. Conclusión de la Fase 1: Implementación de Algoritmos

La primera fase experimental del proyecto confirma que la implementación de los códigos Hamming (7,4) y (15,11) en el dominio espacial es computacionalmente precisa. Ambos algoritmos logran ocultar flujos de bits minimizando la tasa de alteración a un máximo estricto de una modificación por bloque, lo cual garantiza la alta eficiencia de incrustación que fundamenta la técnica de Matrix Embedding.

Al validar que la matemática de los síndromes opera correctamente y que los algoritmos manejan la inversión de los bits menos significativos (LSB) sin inducir errores de rango en la escala de grises, se establece una base algorítmica robusta. Esto permite dar por concluida la implementación criptográfica y avanzar con garantías hacia el procesamiento de imágenes a gran escala, la generación del corpus experimental y la posterior confrontación contra el esteganálisis basado en redes residuales.

## 4. Implementación de Matrix Embedding para generar el dataset

A partir del fundamento matemático anterior, esta sección implementa las funciones que realmente se usan para transformar cada imagen cover en sus versiones esteganográficas. 

Se generan dos variantes:

- `stego_7`: imágenes con mensaje oculto usando Hamming (7,4).
- `stego_15`: imágenes con mensaje oculto usando Hamming (15,11).

La generación se hace de forma determinística por imagen para que el dataset pueda reproducirse usando la misma semilla global.

**Nota importante** Las funciones `embed_hamming_7_3` y `embed_hamming_15_11` se redefinen en esta sección en una versión compacta que usa las matrices globales `H_7_3` y `H_15_11`. Por tanto, estas definiciones sobrescriben en el kernel las versiones didácticas presentadas en la Sección 3. La función usada en el pipeline real de generación es `aplicar_matrix_embedding_vectorizado`, no las versiones escalares.

In [3]:
# ============================================================
# MATRICES DE COMPROBACIÓN DE PARIDAD
# ============================================================

H_7_3 = np.array([
    [0, 0, 0, 1, 1, 1, 1],
    [0, 1, 1, 0, 0, 1, 1],
    [1, 0, 1, 0, 1, 0, 1]
], dtype=np.uint8)

H_15_11 = np.array([
    [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1],
    [0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1],
    [0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1],
    [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1]
], dtype=np.uint8)


def embed_hamming_7_3(pixels_block, secret_bits):
    """
    Versión escalar de referencia para ocultar 3 bits en un bloque de 7 píxeles
    usando Matrix Embedding.

    Esta función no se invoca en el pipeline de generación del dataset. En la
    generación real, la funcionalidad fue reemplazada por
    `aplicar_matrix_embedding_vectorizado`, que procesa bloques de la imagen de
    forma vectorizada.
    """
    x = pixels_block % 2
    s_current = np.dot(H_7_3, x) % 2
    d = (secret_bits - s_current) % 2
    index_to_change = int(d[0] * 4 + d[1] * 2 + d[2])

    modified_pixels = pixels_block.copy()

    if index_to_change > 0:
        idx = index_to_change - 1
        if modified_pixels[idx] % 2 == 0:
            modified_pixels[idx] += 1
        else:
            modified_pixels[idx] -= 1

    return modified_pixels


def embed_hamming_15_11(pixels_block, secret_bits):
    """
    Oculta 4 bits en un bloque de 15 píxeles usando Matrix Embedding.
    """
    x = pixels_block % 2
    s_current = np.dot(H_15_11, x) % 2
    d = (secret_bits - s_current) % 2
    index_to_change = int(d[0] * 8 + d[1] * 4 + d[2] * 2 + d[3])

    modified_pixels = pixels_block.copy()

    if index_to_change > 0:
        idx = index_to_change - 1
        if modified_pixels[idx] % 2 == 0:
            modified_pixels[idx] += 1
        else:
            modified_pixels[idx] -= 1

    return modified_pixels


def crear_rng_por_imagen(base_name, esquema, seed_global=42):
    """
    Crea un generador pseudoaleatorio determinístico por imagen y esquema.
    Esto garantiza reproducibilidad aunque cambie el orden de procesamiento.
    """
    clave = f"{seed_global}|{esquema}|{base_name}".encode("utf-8")
    digest = hashlib.sha256(clave).digest()
    seed = int.from_bytes(digest[:4], byteorder="little", signed=False)
    return np.random.default_rng(seed)


def aplicar_matrix_embedding_vectorizado(img, n, k_bits, H, pesos_binarios, rng):
    """
    Aplica Matrix Embedding a una imagen completa de forma vectorizada.

    Parámetros:
    - img: imagen 2D uint8 en escala de grises.
    - n: tamaño del bloque de píxeles, 7 o 15.
    - k_bits: cantidad de bits ocultos por bloque, 3 o 4.
    - H: matriz de comprobación de paridad.
    - pesos_binarios: vector de pesos posicionales para convertir el vector de diferencia de síndromes en un índice decimal (e.g. [4, 2, 1] para k=3, [8, 4, 2, 1] para k=4).
    - rng: generador aleatorio de NumPy.

    Retorna:
    - Imagen stego 2D uint8.
    """
    if img.ndim != 2:
        raise ValueError("La imagen debe estar en escala de grises con forma (alto, ancho).")

    flat = img.reshape(-1).astype(np.uint8, copy=True)

    num_blocks = len(flat) // n
    if num_blocks == 0:
        return flat.reshape(img.shape)

    usable = num_blocks * n
    blocks = flat[:usable].reshape(num_blocks, n)

    secret = rng.integers(0, 2, size=(num_blocks, k_bits), dtype=np.uint8)

    lsb = blocks % 2
    syndrome = (lsb @ H.T) % 2
    d = (secret - syndrome) % 2

    indices = (d @ pesos_binarios).astype(np.int16)
    mask = indices > 0

    if np.any(mask):
        filas = np.where(mask)[0]
        columnas = indices[mask] - 1

        valores = blocks[filas, columnas].astype(np.int16)

        # Los valores se castearon a int16 para evitar underflow/overflow de uint8 (p.ej. 0−1=255 en uint8). Aquí se invierte el LSB de forma segura.
        nuevos_valores = np.where(valores % 2 == 0, valores + 1, valores - 1)
        blocks[filas, columnas] = nuevos_valores.astype(np.uint8)

    return flat.reshape(img.shape)


def generar_stego_7(img, base_name, seed_global=42):
    rng = crear_rng_por_imagen(base_name, "stego_7", seed_global)
    return aplicar_matrix_embedding_vectorizado(
        img=img,
        n=7,
        k_bits=3,
        H=H_7_3,
        pesos_binarios=np.array([4, 2, 1], dtype=np.uint8),
        rng=rng
    )


def generar_stego_15(img, base_name, seed_global=42):
    rng = crear_rng_por_imagen(base_name, "stego_15", seed_global)
    return aplicar_matrix_embedding_vectorizado(
        img=img,
        n=15,
        k_bits=4,
        H=H_15_11,
        pesos_binarios=np.array([8, 4, 2, 1], dtype=np.uint8),
        rng=rng
    )

## 5. Funciones de descarga, extracción y split por imagen base

La función de split replica la lógica del notebook de entrenamiento en PyTorch: primero toma los stems comunes, los mezcla con `SEMILLA = 42`, separa test, luego validación y finalmente entrenamiento.

In [4]:
# ============================================================
# UTILIDADES DE DESCARGA Y PREPARACIÓN
# ============================================================

def natural_key(path):
    """
    Orden natural para seleccionar las primeras N imágenes de manera intuitiva:
    1, 2, 3, ..., 10, 11, ...
    """
    nombre = Path(path).name
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", nombre)]


def descargar_con_progreso(url, destino):
    """
    Descarga un archivo usando urllib, compatible con notebooks locales.
    """
    destino = Path(destino)

    def hook(bloques, tam_bloque, tam_total):
        if tam_total > 0:
            descargado = bloques * tam_bloque
            porcentaje = min(100, descargado * 100 / tam_total)
            print(f"\rDescargando: {porcentaje:6.2f}%", end="")

    urllib.request.urlretrieve(url, destino, reporthook=hook)
    print("\nDescarga finalizada.")


def extraer_zip(zip_path, extract_dir):
    zip_path = Path(zip_path)
    extract_dir = Path(extract_dir)

    print("Extrayendo archivo ZIP...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_dir)

    print("Extracción finalizada.")


def localizar_imagenes_bossbase(extract_dir):
    """
    Localiza todas las imágenes .pgm dentro de la carpeta extraída.
    Se usa rglob para tolerar variaciones en la estructura interna del ZIP.
    """
    extract_dir = Path(extract_dir)

    rutas = [
        p for p in extract_dir.rglob("*.pgm")
        if "__MACOSX" not in str(p)
    ]

    rutas = sorted(rutas, key=natural_key)

    if len(rutas) == 0:
        raise FileNotFoundError(
            f"No se encontraron imágenes .pgm dentro de {extract_dir.resolve()}."
        )

    return rutas


def dividir_stems_sin_solapamiento(stems, validation_split=0.25, test_split=0.15, seed=42):
    """
    Divide identificadores de imagen base sin solapamiento entre train, val y test.

    Esta función replica la lógica del notebook de entrenamiento en PyTorch.

    IMPORTANTE: los valores de validation_split, test_split y seed deben ser
    idénticos a los usados en hamming_embedding_resnet_steganalysis.ipynb. Si se
    modifican en uno de los notebooks, deben actualizarse en el otro.
    """
    rng = np.random.default_rng(seed)
    stems = np.array(sorted(stems))
    rng.shuffle(stems)

    n_total = len(stems)
    if n_total < 3:
        raise ValueError("Se necesitan al menos 3 imágenes base para crear train, val y test.")

    n_test = max(1, int(round(n_total * test_split))) if test_split > 0 else 0
    n_val = max(1, int(round(n_total * validation_split))) if validation_split > 0 else 0

    if n_test + n_val >= n_total:
        n_test = max(1, min(n_test, n_total - 2))
        n_val = max(1, min(n_val, n_total - n_test - 1))

    stems_test = stems[:n_test]
    stems_val = stems[n_test:n_test + n_val]
    stems_train = stems[n_test + n_val:]

    assert len(set(stems_train) & set(stems_val)) == 0
    assert len(set(stems_train) & set(stems_test)) == 0
    assert len(set(stems_val) & set(stems_test)) == 0

    return list(stems_train), list(stems_val), list(stems_test)


def guardar_lista(path, valores):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(map(str, valores)), encoding="utf-8")


def leer_lista(path):
    path = Path(path)
    return [line.strip() for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

## 6. Descarga y extracción de BOSSBase 1.01

Esta celda descarga el ZIP oficial solo si no existe previamente. Si ya existe `bossbase.zip`, se reutiliza.

In [5]:
# ============================================================
# DESCARGA Y EXTRACCIÓN
# ============================================================

if not ZIP_PATH.exists():
    print("Descargando BOSSBase 1.01 desde el repositorio oficial...")
    descargar_con_progreso(URL_BOSSBASE, ZIP_PATH)
else:
    print(f"El archivo {ZIP_PATH} ya existe. Se omite la descarga.")

# Extraer si la carpeta no existe o si no contiene imágenes .pgm.
debe_extraer = True
if EXTRACT_DIR.exists():
    try:
        rutas_existentes = localizar_imagenes_bossbase(EXTRACT_DIR)
        debe_extraer = len(rutas_existentes) == 0
    except FileNotFoundError:
        debe_extraer = True

if debe_extraer:
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    extraer_zip(ZIP_PATH, EXTRACT_DIR)
else:
    print(f"La carpeta {EXTRACT_DIR} ya contiene imágenes .pgm. Se omite la extracción.")

rutas_bossbase = localizar_imagenes_bossbase(EXTRACT_DIR)
print(f"Imágenes .pgm encontradas: {len(rutas_bossbase)}")

if CANTIDAD_A_PROCESAR > len(rutas_bossbase):
    raise ValueError(
        f"CANTIDAD_A_PROCESAR={CANTIDAD_A_PROCESAR}, "
        f"pero solo hay {len(rutas_bossbase)} imágenes disponibles."
    )

rutas_seleccionadas = rutas_bossbase[:CANTIDAD_A_PROCESAR]
base_names = [p.stem for p in rutas_seleccionadas]

if len(base_names) != len(set(base_names)):
    raise ValueError("Hay nombres base duplicados en las imágenes seleccionadas.")

print(f"Imágenes seleccionadas para el experimento: {len(rutas_seleccionadas)}")
print(f"Primer stem: {base_names[0]} | Último stem seleccionado: {base_names[-1]}")

Descargando BOSSBase 1.01 desde el repositorio oficial...
Descargando: 100.00%
Descarga finalizada.
Extrayendo archivo ZIP...
Extracción finalizada.
Imágenes .pgm encontradas: 10000
Imágenes seleccionadas para el experimento: 1000
Primer stem: 1 | Último stem seleccionado: 1000


## 7. Split por imagen base

El split se hace antes de construir los pares `cover/stego`. Esto evita que una imagen base aparezca en más de una partición.

Aunque las imágenes se guardan físicamente en carpetas por clase (`cover`, `stego_7`, `stego_15`), los archivos `.txt` de `splits/` documentan qué stems pertenecen a train, validation y test.

In [ ]:
# ============================================================
# SPLIT SIN DATA LEAKAGE
# ============================================================

if (
    not RECONSTRUIR_DATASET
    and TRAIN_SPLIT_PATH.exists()
    and VAL_SPLIT_PATH.exists()
    and TEST_SPLIT_PATH.exists()
):
    print("Se encontraron splits existentes. Se reutilizan para mantener reproducibilidad.")

    stems_train = leer_lista(TRAIN_SPLIT_PATH)
    stems_val = leer_lista(VAL_SPLIT_PATH)
    stems_test = leer_lista(TEST_SPLIT_PATH)

else:
    print("Creando nuevos splits por imagen base...")

    stems_train, stems_val, stems_test = dividir_stems_sin_solapamiento(
        stems=base_names,
        validation_split=VALIDATION_SPLIT,
        test_split=TEST_SPLIT,
        seed=SEMILLA
    )

    SPLIT_DIR.mkdir(parents=True, exist_ok=True)

    guardar_lista(TRAIN_SPLIT_PATH, stems_train)
    guardar_lista(VAL_SPLIT_PATH, stems_val)
    guardar_lista(TEST_SPLIT_PATH, stems_test)

train_set = set(stems_train)
val_set = set(stems_val)
test_set = set(stems_test)

assert train_set.isdisjoint(val_set)
assert train_set.isdisjoint(test_set)
assert val_set.isdisjoint(test_set)

print("Split por imagen base listo:")
print(f"Train: {len(stems_train)} imágenes base")
print(f"Val:   {len(stems_val)} imágenes base")
print(f"Test:  {len(stems_test)} imágenes base")
print("Solapamiento train/val/test: 0")

print("\nManifiestos:")
print(TRAIN_SPLIT_PATH)
print(VAL_SPLIT_PATH)
print(TEST_SPLIT_PATH)

Split por imagen base creado correctamente:
Train: 600 imágenes base
Val:   250 imágenes base
Test:  150 imágenes base
Solapamiento train/val/test: 0

Manifiestos guardados en:
splits\train_base_names.txt
splits\val_base_names.txt
splits\test_base_names.txt


## 8. Generación de la carpeta `dataset/`

Esta celda crea la estructura final esperada por el notebook de entrenamiento:

```text
dataset/
├── cover/
├── stego_7/
└── stego_15/
```

La carpeta `cover/` contiene las imágenes originales de BOSSBase. Las carpetas `stego_7/` y `stego_15/` contienen las versiones modificadas mediante Matrix Embedding.

In [ ]:
# ============================================================
# GENERACIÓN DEL DATASET CLASIFICADO PARA PYTORCH
# ============================================================

dataset_existe = (
    COVER_DIR.exists()
    and STEGO_7_DIR.exists()
    and STEGO_15_DIR.exists()
    and len(list(COVER_DIR.glob("*"))) > 0
    and len(list(STEGO_7_DIR.glob("*"))) > 0
    and len(list(STEGO_15_DIR.glob("*"))) > 0
)

if dataset_existe and not RECONSTRUIR_DATASET:
    print("El dataset ya existe y RECONSTRUIR_DATASET=False.")
    print("Se omite la generación de imágenes para evitar reprocesamiento innecesario.")

else:
    if RECONSTRUIR_DATASET and DATASET_DIR.exists():
        print("Eliminando dataset anterior...")
        shutil.rmtree(DATASET_DIR)

    COVER_DIR.mkdir(parents=True, exist_ok=True)
    STEGO_7_DIR.mkdir(parents=True, exist_ok=True)
    STEGO_15_DIR.mkdir(parents=True, exist_ok=True)

    rutas_por_stem = {p.stem: p for p in rutas_seleccionadas}

    particiones = [
        ("train", stems_train),
        ("val", stems_val),
        ("test", stems_test),
    ]

    inicio = time.time()
    total_procesadas = 0
    total_a_procesar = sum(len(stems) for _, stems in particiones)

    print("Iniciando generación del dataset clasificado...")
    print(f"Total de imágenes base a procesar: {total_a_procesar}")

    for nombre_particion, stems_particion in particiones:
        print(f"\nProcesando partición lógica: {nombre_particion} ({len(stems_particion)} imágenes base)")

        for i, stem in enumerate(stems_particion, start=1):
            src_path = rutas_por_stem[stem]

            img = cv2.imread(str(src_path), cv2.IMREAD_GRAYSCALE)
            if img is None:
                raise ValueError(f"No se pudo leer la imagen fuente: {src_path}")

            # 1. Guardar imagen cover
            if FORMATO_COVER == "pgm":
                cover_out = COVER_DIR / f"{stem}.pgm"
                shutil.copy2(src_path, cover_out)
            else:
                cover_out = COVER_DIR / f"{stem}.png"
                cv2.imwrite(str(cover_out), img)

            # 2. Generar stego con Hamming (7,4)
            stego_7 = generar_stego_7(img, base_name=stem, seed_global=SEMILLA)
            stego_7_out = STEGO_7_DIR / f"{stem}.png"
            cv2.imwrite(str(stego_7_out), stego_7)

            # 3. Generar stego con Hamming (15,11)
            stego_15 = generar_stego_15(img, base_name=stem, seed_global=SEMILLA)
            stego_15_out = STEGO_15_DIR / f"{stem}.png"
            cv2.imwrite(str(stego_15_out), stego_15)

            total_procesadas += 1

            if i % 100 == 0 or i == len(stems_particion):
                print(
                    f"  {nombre_particion}: {i}/{len(stems_particion)} "
                    f"| total: {total_procesadas}/{total_a_procesar}"
                )

    duracion = time.time() - inicio

    print("\nGeneración finalizada.")
    print(f"Tiempo total: {duracion / 60:.2f} minutos")

Iniciando generación del dataset clasificado...
Total de imágenes base a procesar: 1000

Procesando partición lógica: train (600 imágenes base)
  train: 100/600 | total: 100/1000
  train: 200/600 | total: 200/1000
  train: 300/600 | total: 300/1000
  train: 400/600 | total: 400/1000
  train: 500/600 | total: 500/1000
  train: 600/600 | total: 600/1000

Procesando partición lógica: val (250 imágenes base)
  val: 100/250 | total: 700/1000
  val: 200/250 | total: 800/1000
  val: 250/250 | total: 850/1000

Procesando partición lógica: test (150 imágenes base)
  test: 100/150 | total: 950/1000
  test: 150/150 | total: 1000/1000

Generación finalizada.
Tiempo total: 0.19 minutos


## 9. Verificación final

Esta celda verifica:

1. Que existan las tres carpetas esperadas.
2. Que las tres clases tengan la misma cantidad de imágenes.
3. Que cada `cover` tenga un par con el mismo stem en `stego_7` y `stego_15`.
4. Que no exista solapamiento entre train, val y test a nivel de imagen base.

In [8]:
# ============================================================
# VERIFICACIÓN FINAL DEL DATASET
# ============================================================

EXTENSIONES_IMG = ["*.png", "*.jpg", "*.jpeg", "*.bmp", "*.pgm", "*.tif", "*.tiff"]

def listar_stems(carpeta):
    stems = set()
    for ext in EXTENSIONES_IMG:
        for ruta in Path(carpeta).glob(ext):
            stems.add(ruta.stem)
    return stems


cover_stems = listar_stems(COVER_DIR)
stego_7_stems = listar_stems(STEGO_7_DIR)
stego_15_stems = listar_stems(STEGO_15_DIR)

print("Conteo final:")
print(f"dataset/cover:    {len(cover_stems)} imágenes")
print(f"dataset/stego_7:  {len(stego_7_stems)} imágenes")
print(f"dataset/stego_15: {len(stego_15_stems)} imágenes")

assert len(cover_stems) == CANTIDAD_A_PROCESAR
assert cover_stems == stego_7_stems, "No todos los cover tienen par en stego_7."
assert cover_stems == stego_15_stems, "No todos los cover tienen par en stego_15."

stems_train_check = set(leer_lista(TRAIN_SPLIT_PATH))
stems_val_check = set(leer_lista(VAL_SPLIT_PATH))
stems_test_check = set(leer_lista(TEST_SPLIT_PATH))

assert stems_train_check.isdisjoint(stems_val_check)
assert stems_train_check.isdisjoint(stems_test_check)
assert stems_val_check.isdisjoint(stems_test_check)

assert stems_train_check | stems_val_check | stems_test_check == cover_stems

print("\nVerificación anti-leakage correcta:")
print(" - No hay imágenes base repetidas entre train, val y test.")
print(" - Los stems de cover, stego_7 y stego_15 coinciden exactamente.")
print(" - La carpeta dataset/ quedó lista para el notebook de entrenamiento en PyTorch.")

print("\nEstructura generada:")
print("dataset/")
print("├── cover/")
print("├── stego_7/")
print("└── stego_15/")
print("\nsplits/")
print("├── train_base_names.txt")
print("├── val_base_names.txt")
print("└── test_base_names.txt")

Conteo final:
dataset/cover:    1000 imágenes
dataset/stego_7:  1000 imágenes
dataset/stego_15: 1000 imágenes

Verificación anti-leakage correcta:
 - No hay imágenes base repetidas entre train, val y test.
 - Los stems de cover, stego_7 y stego_15 coinciden exactamente.
 - La carpeta dataset/ quedó lista para el notebook de entrenamiento en PyTorch.

Estructura generada:
dataset/
├── cover/
├── stego_7/
└── stego_15/

splits/
├── train_base_names.txt
├── val_base_names.txt
└── test_base_names.txt


## 10. Evaluación de fidelidad visual del dataset generado

Para medir el impacto del *Matrix Embedding* en las imágenes de cobertura, se emplearon dos métricas estándar en el procesamiento de imágenes esteganográficas:

1. PSNR (Peak Signal-to-Noise Ratio): Mide la relación señal-ruido de pico entre la imagen de cobertura original y la imagen esteganográfica. Se calcula comparando el máximo valor teórico de la señal contra el Error Cuadrático Medio (MSE) entre ambas imágenes. Se expresa en decibelios (dB). Un valor más alto indica una menor degradación visual. Su formulación matemática es:

$$MSE = \frac{1}{m\,n} \sum_{i=0}^{m-1} \sum_{j=0}^{n-1} [I(i,j) - K(i,j)]^2$$
$$PSNR = 10 \cdot \log_{10} \left( \frac{MAX_I^2}{MSE} \right)$$

2. SSIM (Structural Similarity Index Measure): A diferencia del PSNR, que mide el error absoluto, el SSIM evalúa la degradación de la estructura espacial de la imagen, modelando mejor la percepción del ojo humano. Sus valores oscilan entre $-1$ y $1$, donde $1$ indica una imagen idéntica a la original.

Estas métricas permiten verificar si el mensaje oculto modifica la imagen de forma perceptiblemente baja. Se espera que el esquema basado en Hamming (15,11) reporte valores de PSNR y SSIM superiores a los de Hamming (7,4). Esta ventaja de fidelidad se debe a que Hamming (15,11) tiene una menor densidad de modificación, pues la probabilidad de alterar un píxel es aproximadamente 1/n y disminuye al usar bloques más grandes. Además, presenta una mayor eficiencia de embedding: oculta 4 bits por modificación, frente a 3 bits por modificación en Hamming (7,4). Por tanto, las tasas de payload absolutas no son equivalentes entre ambos esquemas.

In [ ]:
# ============================================================
# EVALUACIÓN DE FIDELIDAD VISUAL: PSNR Y SSIM
# ============================================================

EXTENSIONES_VALIDAS = [".png", ".jpg", ".jpeg", ".bmp", ".pgm", ".tif", ".tiff"]

def listar_imagenes_por_stem(carpeta):
    """
    Retorna un diccionario {stem: ruta_imagen}.
    Permite comparar cover y stego aunque tengan extensiones distintas.
    """
    carpeta = Path(carpeta)
    rutas = {}

    for path in carpeta.iterdir():
        if path.suffix.lower() in EXTENSIONES_VALIDAS:
            rutas[path.stem] = path

    return rutas


def evaluar_fidelidad_visual(ruta_cover, ruta_stego, nombre_esquema, limite=None, seed=42):
    """
    Compara imágenes cover y stego por nombre base.
    Retorna un DataFrame con PSNR y SSIM por imagen.
    """
    cover_por_stem = listar_imagenes_por_stem(ruta_cover)
    stego_por_stem = listar_imagenes_por_stem(ruta_stego)

    stems_comunes = sorted(set(cover_por_stem.keys()) & set(stego_por_stem.keys()))

    if len(stems_comunes) == 0:
        raise ValueError(f"No hay imágenes comunes entre {ruta_cover} y {ruta_stego}.")

    if limite is not None and limite < len(stems_comunes):
        rng = np.random.default_rng(seed)
        stems_comunes = sorted(rng.choice(stems_comunes, size=limite, replace=False))

    resultados = []

    for stem in stems_comunes:
        img_cover = cv2.imread(str(cover_por_stem[stem]), cv2.IMREAD_GRAYSCALE)
        img_stego = cv2.imread(str(stego_por_stem[stem]), cv2.IMREAD_GRAYSCALE)

        if img_cover is None or img_stego is None:
            continue

        if img_cover.shape != img_stego.shape:
            raise ValueError(
                f"Las imágenes no tienen la misma forma para stem={stem}: "
                f"{img_cover.shape} vs {img_stego.shape}"
            )

        psnr = compute_psnr(img_cover, img_stego, data_range=255)
        ssim = compute_ssim(img_cover, img_stego, data_range=255, win_size=11)

        pixeles_modificados = np.sum(img_cover != img_stego)
        total_pixeles = img_cover.size
        porcentaje_modificado = pixeles_modificados / total_pixeles * 100

        resultados.append({
            "esquema": nombre_esquema,
            "stem": stem,
            "psnr": psnr,
            "ssim": ssim,
            "pixeles_modificados": pixeles_modificados,
            "total_pixeles": total_pixeles,
            "porcentaje_modificado": porcentaje_modificado
        })

    return pd.DataFrame(resultados)


df_fidelidad_7 = evaluar_fidelidad_visual(
    ruta_cover=COVER_DIR,
    ruta_stego=STEGO_7_DIR,
    nombre_esquema="Hamming (7,4)",
    limite=MUESTRAS_EVALUACION,
    seed=SEMILLA
)

df_fidelidad_15 = evaluar_fidelidad_visual(
    ruta_cover=COVER_DIR,
    ruta_stego=STEGO_15_DIR,
    nombre_esquema="Hamming (15,11)",
    limite=MUESTRAS_EVALUACION,
    seed=SEMILLA
)

df_fidelidad = pd.concat([df_fidelidad_7, df_fidelidad_15], ignore_index=True)

resumen_fidelidad = (
    df_fidelidad
    .groupby("esquema")
    .agg(
        psnr_promedio=("psnr", "mean"),
        psnr_std=("psnr", "std"),
        ssim_promedio=("ssim", "mean"),
        ssim_std=("ssim", "std"),
        porcentaje_modificado_promedio=("porcentaje_modificado", "mean")
    )
    .reset_index()
)

print(f"Resultados de fidelidad visual sobre {MUESTRAS_EVALUACION} imágenes:")
display(resumen_fidelidad)

df_fidelidad.to_csv(FIDELITY_RESULTS_PATH, index=False)
print(f"Resultados detallados guardados en: {FIDELITY_RESULTS_PATH}")

psnr_7 = resumen_fidelidad.loc[
    resumen_fidelidad["esquema"] == "Hamming (7,4)",
    "psnr_promedio"
].iloc[0]

psnr_15 = resumen_fidelidad.loc[
    resumen_fidelidad["esquema"] == "Hamming (15,11)",
    "psnr_promedio"
].iloc[0]

if psnr_15 > psnr_7:
    print("\nLa hipótesis se confirma: Hamming (15,11) produce menor distorsión visual promedio.")
else:
    print("\nLa hipótesis no se confirma para esta muestra: Hamming (15,11) no obtuvo mayor PSNR promedio.")

In [ ]:
# ============================================================
# GRÁFICA COMPARATIVA DE FIDELIDAD VISUAL
# ============================================================

plt.figure(figsize=(7, 5))
df_fidelidad.boxplot(column="psnr", by="esquema")
plt.title("Comparación de PSNR por esquema de Hamming")
plt.suptitle("")
plt.xlabel("Esquema")
plt.ylabel("PSNR [dB]")
plt.grid(True)
plt.show()

plt.figure(figsize=(7, 5))
df_fidelidad.boxplot(column="ssim", by="esquema")
plt.title("Comparación de SSIM por esquema de Hamming")
plt.suptitle("")
plt.xlabel("Esquema")
plt.ylabel("SSIM")
plt.grid(True)
plt.show()

## 11. Metadata del dataset

Se guarda un archivo JSON con los parámetros usados para que el experimento sea reproducible.

In [ ]:
# ============================================================
# GUARDAR METADATA DEL DATASET
# ============================================================

metadata = {
    "dataset_dir": str(DATASET_DIR.resolve()),
    "bossbase_url": URL_BOSSBASE,
    "cantidad_a_procesar": CANTIDAD_A_PROCESAR,
    "validation_split": VALIDATION_SPLIT,
    "test_split": TEST_SPLIT,
    "train_base_images": len(stems_train),
    "val_base_images": len(stems_val),
    "test_base_images": len(stems_test),
    "seed": SEMILLA,
    "cover_format": FORMATO_COVER,
    "output_structure": {
        "cover": str(COVER_DIR),
        "stego_7": str(STEGO_7_DIR),
        "stego_15": str(STEGO_15_DIR),
        "splits": str(SPLIT_DIR),
    },
    "fidelity_results": str(FIDELITY_RESULTS_PATH),
    "anti_leakage_rule": (
        "El split se realiza sobre el identificador de imagen base antes de usar los pares cover/stego."
    ),
    "hamming_schemes": {
        "stego_7": {
            "code": "Hamming (7,4)",
            "pixels_per_block": 7,
            "secret_bits_per_block": 3
        },
        "stego_15": {
            "code": "Hamming (15,11)",
            "pixels_per_block": 15,
            "secret_bits_per_block": 4
        }
    }
}

if "resumen_fidelidad" in globals():
    metadata["fidelity_summary"] = resumen_fidelidad.to_dict(orient="records")

METADATA_PATH.write_text(
    json.dumps(metadata, indent=4, ensure_ascii=False),
    encoding="utf-8"
)

print(f"Metadata guardada en: {METADATA_PATH}")
print(json.dumps(metadata, indent=4, ensure_ascii=False))

Metadata guardada en: dataset_metadata_pytorch.json
{
    "dataset_dir": "C:\\Users\\m.amorocho\\proyecto-cripto\\dataset",
    "bossbase_url": "http://dde.binghamton.edu/download/ImageDB/BOSSbase_1.01.zip",
    "cantidad_a_procesar": 1000,
    "validation_split": 0.25,
    "test_split": 0.15,
    "train_base_images": 600,
    "val_base_images": 250,
    "test_base_images": 150,
    "seed": 42,
    "cover_format": "pgm",
    "output_structure": {
        "cover": "dataset\\cover",
        "stego_7": "dataset\\stego_7",
        "stego_15": "dataset\\stego_15",
        "splits": "splits"
    },
    "anti_leakage_rule": "El split se realiza sobre el identificador de imagen base antes de usar los pares cover/stego."
}


## 12. Uso con el notebook de entrenamiento

Después de ejecutar este notebook, el notebook `hamming_embedding_resnet_steganalysis.ipynb` puede usar la carpeta generada directamente.

Si ambos notebooks están en la misma carpeta, basta con dejar:

```python
RUTA_PROYECTO = "."
```

El entrenamiento leerá:

```python
dataset/cover
dataset/stego_7
dataset/stego_15
```

y su función `crear_dataloaders_binarios()` volverá a dividir por `stem`, no por archivo individual, evitando data leakage.